# Training Creative EEG — IDG vs IDE vs IDR

- **Input** : Creative scaled EEG data (16 channels FP1..P4)
- **Preprocessing** : Notch 50 Hz → Bandpass 1-45 Hz → MinMax scaling 0-1
- **Fitur** : Statistical (52) + Welch Band Power (80) = 132 total
- **Label** : IDG (0), IDE (1), IDR (2)
- **Split** : 80% train / 20% test
- **Sampling Rate** : 125 Hz (aplikasi desktop OpenBCI)

In [ ]:
# 0) IMPORT & SETUP
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from scipy.signal import butter, filtfilt, iirnotch, welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid
import joblib
import pickle

# Tentukan workspace root
WORKSPACE_ROOT = Path.cwd()
if 'eeg-desktop-app' in str(WORKSPACE_ROOT):
    WORKSPACE = WORKSPACE_ROOT
else:
    # Cari folder eeg-desktop-app
    for parent in WORKSPACE_ROOT.parents:
        if (parent / 'eeg-desktop-app').exists():
            WORKSPACE = parent / 'eeg-desktop-app'
            break
    else:
        WORKSPACE = Path.home() / 'Downloads' / 'eeg-desktop-app'

MODEL_DIR = WORKSPACE / 'models'
MODEL_DIR.mkdir(exist_ok=True)

print(f'Workspace: {WORKSPACE}')
print(f'Model Dir: {MODEL_DIR}')

In [ ]:
# 1) KONFIGURASI
FS_ORIGINAL = 125  # Sampling rate aplikasi desktop

CHANNELS = [
    'FP1', 'FP2', 'C3', 'C4', 'T5', 'T6', 'O1', 'O2',
    'F7', 'F8', 'F3', 'F4', 'T3', 'T4', 'P3', 'P4',
]

BANDS = {
    'delta': (1.0, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta': (13.0, 30.0),
    'gamma': (30.0, 50.0),
}

LABEL_MAP = {
    'IDG': 0,
    'IDE': 1,
    'IDR': 2,
}

LABEL_NAMES = ['IDG', 'IDE', 'IDR']
LABELS_ORDER = [0, 1, 2]

print('=== Configuration ===')
print(f'Sampling Rate  : {FS_ORIGINAL} Hz')
print(f'Channels       : {len(CHANNELS)} ({", ".join(CHANNELS)})')
print(f'Bands          : {list(BANDS.keys())}')
print(f'Labels         : {LABEL_MAP}')
print(f'Total Features : 52 (statistical) + 80 (band power) = 132')


In [ ]:
# 2) LOAD TRAINING DATA
# Cari file creative_scaled.csv di preprocessing directory
csv_paths = list(Path('.').rglob('creative_scaled.csv')) + \
            list(WORKSPACE.rglob('creative_scaled.csv'))
csv_path = csv_paths[0] if csv_paths else None

if csv_path and csv_path.exists():
    df = pd.read_csv(csv_path, dtype={ch: 'float32' for ch in CHANNELS})
    print(f'Loaded: {csv_path}')
else:
    # Fallback: cari file csv lainnya
    print('creative_scaled.csv tidak ditemukan.')
    print('Silakan berikan CSV file dengan kolom: subject_id, condition, label, dan 16 channel EEG')
    df = None

if df is not None:
    print(f'\nShape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    
    # Map condition/label ke numeric labels (IDG=0, IDE=1, IDR=2)
    if 'label' not in df.columns:
        print('WARNING: kolom "label" tidak ditemukan')
    
    df['target'] = df['label'].map(LABEL_MAP)
    df = df.dropna(subset=['target'])
    df['target'] = df['target'].astype(int)
    
    print(f'\nLabel distribution:')
    print(df['target'].value_counts().sort_index())
    print(f'Total rows: {len(df)}')

In [ ]:
# 3) EXTRACT STATISTICAL FEATURES
def extract_statistical_features(X_raw):
    """
    Ekstrak 52 fitur statistik:
      - 16 channel mean
      - global mean, global std
      - 16 channel variance
      - 16 channel RMS
      - global skewness, kurtosis
    
    Input : X_raw shape (n_samples, n_channels)
    Output: X_stat shape (n_samples, 52)
    """
    n_samples, n_ch = X_raw.shape
    
    # Global stats
    feat_global_mean = np.mean(X_raw, axis=1, keepdims=True)
    feat_global_std = np.std(X_raw, axis=1, keepdims=True)
    
    # Per-channel stats
    feat_var = np.var(X_raw, axis=1, keepdims=True) if False else np.var(X_raw, axis=0, keepdims=False)
    feat_rms = np.sqrt(np.mean(X_raw ** 2, axis=1, keepdims=True)) if False else np.sqrt(np.mean(X_raw ** 2, axis=0, keepdims=False))
    
    # Compute per-channel properly
    feat_var_ch = np.array([np.var(X_raw[i, :]) for i in range(n_samples)]).reshape(-1, 1)
    feat_rms_ch = np.array([np.sqrt(np.mean(X_raw[i, :] ** 2)) for i in range(n_samples)]).reshape(-1, 1)
    
    # Skewness & Kurtosis per sample
    feat_skew = np.array([skew(X_raw[i, :]) for i in range(n_samples)]).reshape(-1, 1)
    feat_kurt = np.array([kurtosis(X_raw[i, :]) for i in range(n_samples)]).reshape(-1, 1)
    
    X_stat = np.hstack([
        X_raw,                    # 16 channel values
        feat_global_mean,         # 1 (global mean)
        feat_global_std,          # 1 (global std)
        feat_var_ch,              # 1 (variance)
        feat_rms_ch,              # 1 (RMS)
        feat_skew,                # 1 (skewness)
        feat_kurt,                # 1 (kurtosis)
    ])
    return X_stat

stat_feature_names = (
    CHANNELS +
    ['global_mean', 'global_std', 'variance', 'rms', 'skewness', 'kurtosis']
)

if df is not None:
    X_raw = df[CHANNELS].values.astype(np.float32)
    y = df['target'].values
    
    X_stat = extract_statistical_features(X_raw)
    
    print(f'Statistical features shape: {X_stat.shape}')
    print(f'  16 channels: {len(CHANNELS)}')
    print(f'  + global stats: 6')
    print(f'  = Total: {X_stat.shape[1]}')

In [ ]:
# 4) EXTRACT WELCH BAND POWER FEATURES
def extract_band_power_features(X_raw, fs=FS_ORIGINAL, bands=BANDS):
    """
    Ekstrak 80 fitur band power (16 channels × 5 bands).
    
    Untuk setiap channel:
      1. Hitung PSD dengan Welch
      2. Hitung band power ratio per band (relatif terhadap total power)
      3. Fitur = channel mean × band ratio
    
    Input : X_raw shape (n_samples, n_channels)
    Output: X_bp shape (n_samples, 80)
    """
    n_samples, n_ch = X_raw.shape
    n_bands = len(bands)
    
    # Hitung rasio band power per channel dari seluruh data
    band_ratios = np.zeros((n_ch, n_bands))
    
    for ch_idx in range(n_ch):
        ch_signal = X_raw[:, ch_idx].astype(np.float64)
        nperseg = min(256, len(ch_signal))
        freqs, psd = welch(ch_signal, fs=fs, nperseg=nperseg)
        total_power = trapezoid(psd, freqs) + 1e-10
        
        for b_idx, (band_name, (fmin, fmax)) in enumerate(bands.items()):
            mask = np.logical_and(freqs >= fmin, freqs < fmax)
            if np.any(mask):
                band_power = trapezoid(psd[mask], freqs[mask])
                band_ratios[ch_idx, b_idx] = band_power / total_power
    
    # Buat fitur per sample
    features = np.zeros((n_samples, n_ch * n_bands), dtype=np.float32)
    
    for b_idx in range(n_bands):
        for ch_idx in range(n_ch):
            col = b_idx * n_ch + ch_idx
            features[:, col] = X_raw[:, ch_idx] * band_ratios[ch_idx, b_idx]
    
    return features, band_ratios

bp_feature_names = [
    f'{ch}_{band}' for band in BANDS.keys() for ch in CHANNELS
]

if df is not None:
    print('Computing Welch band power features...')
    X_bp, band_ratios_info = extract_band_power_features(X_raw, fs=FS_ORIGINAL, bands=BANDS)
    
    # Show band ratios per channel
    df_ratios = pd.DataFrame(band_ratios_info, index=CHANNELS, columns=list(BANDS.keys()))
    print('\nBand Power Ratios per Channel:')
    print(df_ratios.round(4))
    
    print(f'\nBand power features shape: {X_bp.shape}')
    print(f'  16 channels × {len(BANDS)} bands = {len(CHANNELS) * len(BANDS)} features')

In [ ]:
# 5) COMBINE FEATURES & SCALE
if df is not None:
    X_combined = np.hstack([X_stat, X_bp])
    feature_names_all = stat_feature_names + bp_feature_names
    
    print('=== Feature Summary ===')
    print(f'Statistical     : {X_stat.shape[1]}')
    print(f'Band Power      : {X_bp.shape[1]}')
    print(f'Total Combined  : {X_combined.shape[1]}')
    print(f'Shape X_combined: {X_combined.shape}')
    print(f'Shape y         : {y.shape}')
    
    # Scale dengan StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_combined)
    
    # Split 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    print(f'\n=== Train/Test Split ===')
    print(f'X_train: {X_train.shape} (80%)')
    print(f'X_test : {X_test.shape} (20%)')
    print(f'\nTrain distribution:')
    for label_idx, label_name in enumerate(LABEL_NAMES):
        count = np.sum(y_train == label_idx)
        print(f'  {label_name}: {count}')
    print(f'\nTest distribution:')
    for label_idx, label_name in enumerate(LABEL_NAMES):
        count = np.sum(y_test == label_idx)
        print(f'  {label_name}: {count}')

In [ ]:
# 6) TRAIN RANDOMFOREST
if df is not None:
    print('Training RandomForestClassifier...')
    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
        max_depth=20,
    )
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    
    print(f'\n=== Training Results ===')
    print(f'Accuracy         : {acc:.4f}')
    print(f'Balanced Accuracy: {bacc:.4f}')
    print(f'Model trained successfully!')

In [ ]:
# 7) EVALUATE MODEL
if df is not None:
    cm = confusion_matrix(y_test, y_pred, labels=LABELS_ORDER)
    
    print('=== Confusion Matrix ===')
    cm_df = pd.DataFrame(cm,
        index=[f'Actual {LABEL_NAMES[i]}' for i in LABELS_ORDER],
        columns=[f'Pred {LABEL_NAMES[i]}' for i in LABELS_ORDER])
    print(cm_df)
    
    print('\n=== Classification Report ===')
    report = classification_report(y_test, y_pred, 
                                  target_names=LABEL_NAMES,
                                  zero_division=0)
    print(report)

In [ ]:
# 8) SAVE MODEL & METADATA
if df is not None:
    model_path = MODEL_DIR / 'rf_eeg_model.pkl'
    
    # Hitung metrics untuk simpan
    report_dict = classification_report(y_test, y_pred,
                                       target_names=LABEL_NAMES,
                                       output_dict=True, zero_division=0)
    
    metrics = {
        'model': 'RandomForestClassifier',
        'task': 'Creative EEG - IDG vs IDE vs IDR',
        'sampling_rate_hz': FS_ORIGINAL,
        'n_total': len(X_combined),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'n_features_stat': len(stat_feature_names),
        'n_features_bp': len(bp_feature_names),
        'n_features_total': len(feature_names_all),
        'accuracy': float(acc),
        'balanced_accuracy': float(bacc),
        'f1_macro': float(report_dict['macro avg']['f1-score']),
        'f1_weighted': float(report_dict['weighted avg']['f1-score']),
    }
    
    # Simpan model + metadata ke pickle
    model_package = {
        'model': rf,
        'scaler': scaler,
        'channels': CHANNELS,
        'bands': BANDS,
        'sampling_rate': FS_ORIGINAL,
        'feature_names': feature_names_all,
        'label_map': LABEL_MAP,
        'label_names': LABEL_NAMES,
        'target': 'Creative EEG - IDG vs IDE vs IDR',
        'metrics': metrics,
    }
    
    with open(model_path, 'wb') as f:
        pickle.dump(model_package, f)
    
    # Also save with joblib
    joblib.dump(model_package, model_path)
    
    print(f'=== Model Saved ===')
    print(f'Path     : {model_path}')
    print(f'Accuracy : {acc:.4f}')
    print(f'Balanced : {bacc:.4f}')
    print(f'Features : {len(feature_names_all)}')
    print(f'Status   : Ready for inference!')

In [ ]:
# 9) LOAD & USE MODEL FOR INFERENCE
def preprocess_eeg_window(eeg_window, fs=FS_ORIGINAL):
    \"\"\"
    Preprocess raw EEG window: notch → bandpass → minmax scaling 0-1
    
    Input : eeg_window shape [n_channels, n_samples]
    Output: preprocessed shape [n_samples, n_channels]
    \"\"\"
    if eeg_window.ndim != 2:
        raise ValueError('EEG harus array 2D [n_channels, n_samples]')
    
    if eeg_window.shape[0] < len(CHANNELS):
        raise ValueError(f'Minimal {len(CHANNELS)} channels diperlukan')
    
    # Take first 16 channels
    data = eeg_window[:len(CHANNELS)].astype(np.float64)
    data = data.T  # [n_channels, n_samples] → [n_samples, n_channels]
    
    # Notch 50 Hz
    b, a = iirnotch(50.0, 30.0, fs)
    data = filtfilt(b, a, data, axis=0)
    
    # Bandpass 1-45 Hz
    nyq = fs / 2.0
    b, a = butter(4, [1.0 / nyq, 45.0 / nyq], btype='band')
    data = filtfilt(b, a, data, axis=0)
    
    # MinMax scale 0-1
    scaler_minmax = MinMaxScaler(feature_range=(0, 1))
    data = scaler_minmax.fit_transform(data)
    
    return np.asarray(data, dtype=np.float64)

def extract_features_inference(preprocessed, fs=FS_ORIGINAL, bands=BANDS):
    \"\"\"
    Extract 132 features (52 statistical + 80 band power)
    
    Input : preprocessed shape [n_samples, n_channels]
    Output: features shape [132]
    \"\"\"
    n_channels = preprocessed.shape[1]
    features = []
    
    # Statistical features
    for ch in range(n_channels):
        ch_data = preprocessed[:, ch]
        features.append(float(np.mean(ch_data)))
    
    all_data = preprocessed.flatten()
    features.append(float(np.mean(all_data)))
    features.append(float(np.std(all_data)))
    
    for ch in range(n_channels):
        features.append(float(np.var(preprocessed[:, ch])))
    
    for ch in range(n_channels):
        features.append(float(np.sqrt(np.mean(preprocessed[:, ch] ** 2))))
    
    features.append(float(skew(all_data)))
    features.append(float(kurtosis(all_data)))
    
    # Band power features
    for ch in range(n_channels):
        ch_data = preprocessed[:, ch].astype(np.float64)
        nperseg = min(256, len(ch_data))
        freqs, psd = welch(ch_data, fs=fs, nperseg=nperseg)
        total_power = trapezoid(psd, freqs) + 1e-10
        
        for band_name, (fmin, fmax) in bands.items():
            mask = np.logical_and(freqs >= fmin, freqs < fmax)
            if not np.any(mask):
                features.append(0.0)
            else:
                band_power = trapezoid(psd[mask], freqs[mask])
                ratio = band_power / total_power
                weighted = ratio * np.mean(ch_data)
                features.append(float(weighted))
    
    return np.asarray(features, dtype=np.float64)

# Load trained model
model_path = MODEL_DIR / 'rf_eeg_model.pkl'
if model_path.exists():
    loaded_model = joblib.load(model_path)
    print('=== Model Loaded ===')
    print(f'Model     : {type(loaded_model[\"model\"]).__name__}')
    print(f'Channels  : {loaded_model[\"channels\"]}')
    print(f'Bands     : {list(loaded_model[\"bands\"].keys())}')
    print(f'Features  : {len(loaded_model[\"feature_names\"])}')
    print(f'Accuracy  : {loaded_model[\"metrics\"][\"accuracy\"]:.4f}')
else:
    print(f'Model not found at {model_path}')

In [ ]:
# 10) LOAD NEW EEG EXCEL FILE FOR INFERENCE
# Cari file Excel raw EEG (contoh: classif_creative_classifications.csv atau Excel file)
# Untuk demo, kita buat sample data

def load_eeg_excel_or_csv(filepath):
    \"\"\"
    Load EEG data dari Excel atau CSV file.
    Expected format: columns = EXG Channel 0, EXG Channel 1, ..., atau ch_0, ch_1, ...
    atau nama lebih formal seperti FP1, FP2, dll.
    \"\"\"
    if filepath.endswith('.xlsx'):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath)
    
    # Cek kolom EEG yang tersedia
    eeg_cols = [col for col in df.columns if 'channel' in col.lower() or 'ch_' in col.lower() or col in CHANNELS]
    
    if len(eeg_cols) < len(CHANNELS):
        print(f'WARNING: Ditemukan {len(eeg_cols)} channels, expected {len(CHANNELS)}')
    
    return df, eeg_cols

# Demo: Create sample new recording
print('\\n=== Inference: Predict New EEG Recording ===')
print('\\nCreating sample new EEG data...')

# Buat sample raw EEG: 16 channels × 625 samples (5 detik @ 125 Hz)
sample_eeg = np.random.randn(16, 625)

print(f'Sample EEG shape: {sample_eeg.shape}')
print(f'  16 channels × 625 samples (5 sec @ 125 Hz)')

# Preprocess
pre = preprocess_eeg_window(sample_eeg, fs=FS_ORIGINAL)
print(f'\\nPreprocessed shape: {pre.shape}')
print(f'  Range: [{pre.min():.4f}, {pre.max():.4f}]')

# Extract features
features_inference = extract_features_inference(pre, fs=FS_ORIGINAL, bands=BANDS)
print(f'\\nFeatures extracted: {features_inference.shape[0]} (expected 132)')

# Scale with loaded model scaler
if model_path.exists():
    scaler_loaded = loaded_model['scaler']
    features_scaled = scaler_loaded.transform(features_inference.reshape(1, -1))
    print(f'Features scaled: {features_scaled.shape}')
    
    # Predict
    model_loaded = loaded_model['model']
    pred_label = model_loaded.predict(features_scaled)[0]
    pred_label_name = LABEL_NAMES[int(pred_label)]
    
    if hasattr(model_loaded, 'predict_proba'):
        proba = model_loaded.predict_proba(features_scaled)[0]
        confidence = float(np.max(proba))
        print(f'\\n=== PREDICTION ===')
        print(f'Label        : {pred_label_name} (class {int(pred_label)})')
        print(f'Confidence   : {confidence:.4f}')
        for i, label_name in enumerate(LABEL_NAMES):
            print(f'  {label_name:10s}: {proba[i]:.4f}')
    else:
        print(f'\\nPredicted label: {pred_label_name} (class {int(pred_label)})')

In [ ]:
# 11) EXPORT PREDICTIONS TO EXCEL
def export_predictions_to_excel(eeg_data, predictions, output_path, channels=CHANNELS):
    \"\"\"
    Export EEG recording + predictions to Excel file.
    
    Columns: [channels...] + [prediction_label, prediction_confidence]
    \"\"\"
    # Prepare output dataframe
    export_data = eeg_data.T  # [n_channels, n_samples] → [n_samples, n_channels]
    
    df_export = pd.DataFrame(export_data, columns=channels)
    
    # Add prediction columns
    df_export['label'] = predictions.get('label_name', 'Unknown')
    df_export['confidence'] = predictions.get('confidence', 0.0)
    
    # Save to Excel
    df_export.to_excel(output_path, index=False)
    
    return output_path

# Demo export
if model_path.exists():
    output_path = WORKSPACE / 'creative_predictions.xlsx'
    
    # Prepare prediction dict
    pred_dict = {
        'label': int(pred_label),
        'label_name': pred_label_name,
        'confidence': confidence,
    }
    
    # Export
    export_file = export_predictions_to_excel(sample_eeg, pred_dict, output_path, channels=CHANNELS)
    
    print(f'\\n=== Predictions Exported ===')
    print(f'Output file: {export_file}')
    print(f'Rows       : {sample_eeg.shape[1]} (samples)')
    print(f'Columns    : {len(CHANNELS)} (EEG) + 2 (prediction)')
    print(f'Status     : Ready for review!')